## Movie Market Analysis
An analytical study of box office performance to provide strategic insights for guiding the development of a new movie studio. This a project from the Moringa Data Science Program.

## Project Objectives
1. Identify the top performing directors and top genres and their effect on the movie market.
2. Identify the top studios and the revenue they generate.
3. To determine which types of movies generate the highest Return on Investment (ROI), using budget and gross revenue data, in order to recommend profitable production strategies for the company new studio.
4. 

In [42]:
import itertools
import numpy as np
import pandas as pd 
from numbers import Number
import sqlite3
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import zipfile
warnings.filterwarnings('ignore')

import pickle

## Data Preparation

In [43]:
cd zippedData

[WinError 2] The system cannot find the file specified: 'zippedData'
C:\Users\PapaDay\Desktop\movies_phase-2\dsc-phase-2-project-v3\zippedData


In [44]:
# Read CSV files compressed with gzip
bom_movie_gross = pd.read_csv('bom.movie_gross.csv.gz',encoding='latin1')

# Read TSV files compressed with gzip
rt_movie_info = pd.read_csv('rt.movie_info.tsv.gz', sep='\t',encoding='latin1')
rt_reviews = pd.read_csv('rt.reviews.tsv.gz', sep='\t',encoding='latin1')

# Read another CSV file compressed with gzip
tmdb_movies = pd.read_csv('tmdb.movies.csv.gz',encoding='latin1')

# Read the movie budgets compressed CSV
tn_movie_budgets = pd.read_csv('tn.movie_budgets.csv.gz',encoding='latin1')


In [45]:
tn_movie_budgets.isnull().sum()

id                   0
release_date         0
movie                0
production_budget    0
domestic_gross       0
worldwide_gross      0
dtype: int64

### Data Cleaning SQL Database

In [46]:
with zipfile.ZipFile('im.db.zip', 'r') as zip_ref:
    zip_ref.extractall('unzipped_db')
## Connect to the .db file
conn = sqlite3.connect('unzipped_db/im.db')
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)

            name
0   movie_basics
1      directors
2      known_for
3     movie_akas
4  movie_ratings
5        persons
6     principals
7        writers


In [47]:
movie_basics="""
SELECT *
FROM movie_basics
"""
movie_basics= pd.read_sql_query(movie_basics,conn)

In [48]:
movie_basics.isnull().sum()

movie_id               0
primary_title          0
original_title        21
start_year             0
runtime_minutes    31739
genres              5408
dtype: int64

In [49]:
movie_basics.shape

(146144, 6)

In [50]:
movie_basics.duplicated().sum()

0

In [51]:
median_runtime = movie_basics["runtime_minutes"].median()
movie_basics["runtime_minutes"] = movie_basics["runtime_minutes"].fillna(median_runtime)

In [52]:
mode_genres = movie_basics["genres"].mode()[0]
movie_basics["genres"] = movie_basics["genres"].fillna(mode_genres)

In [53]:
movie_basics["original_title"] = movie_basics["original_title"].fillna("Unknown")

In [54]:
movie_basics.isnull().sum()

movie_id           0
primary_title      0
original_title     0
start_year         0
runtime_minutes    0
genres             0
dtype: int64

In [58]:
directors="""
SELECT *
FROM directors
"""
directors= pd.read_sql_query(directors,conn)

In [59]:
directors.isnull().sum()

movie_id     0
person_id    0
dtype: int64

In [61]:
movie_ratings="""
SELECT *
FROM movie_ratings
"""
movie_ratings= pd.read_sql_query(movie_ratings,conn)

In [62]:
movie_ratings.isnull().sum()

movie_id         0
averagerating    0
numvotes         0
dtype: int64

## Exploratory Data Analysis

On this section we will be merging tables and relating them to our objectives.
Let's start with directors and ratings of the movies they produce to see top performing directors

In [70]:
# 2.1 Directors and their movies
query_directors = """
SELECT d.movie_id, p.primary_name AS director_name
FROM directors AS d
JOIN persons AS p
ON d.person_id = p.person_id;
"""
directors_df = pd.read_sql_query(query_directors, conn)

# 2.2 Movies and their average ratings
query_ratings = """
SELECT movie_id, averagerating
FROM movie_ratings;
"""
ratings_df = pd.read_sql_query(query_ratings, conn)

# 3. Merge Directors with Ratings
directors_with_ratings = pd.merge(directors_df, ratings_df, on='movie_id')

# 4. Calculate average rating per director
director_avg_rating = directors_with_ratings.groupby('director_name')['averagerating'].mean().sort_values(ascending=False)

# Optional: view top 10 directors by average rating
top_directors = director_avg_rating.head(10)
print(top_directors)

# Define "top directors" (avg rating >= 7.0) ---
top_directors = director_avg_rating[director_avg_rating >= 7.0].index.tolist()
top_directors

director_name
Emre Oran                 10.0
Chad Carpenter            10.0
Loreto Di Cesare          10.0
Michiel Brongers          10.0
Masahiro Hayakawa         10.0
Lindsay Thompson          10.0
Ivana Diniz               10.0
Tristan David Luciotti    10.0
Stephen Peek              10.0
Andrew Jezard              9.9
Name: averagerating, dtype: float64


['Emre Oran',
 'Chad Carpenter',
 'Loreto Di Cesare',
 'Michiel Brongers',
 'Masahiro Hayakawa',
 'Lindsay Thompson',
 'Ivana Diniz',
 'Tristan David Luciotti',
 'Stephen Peek',
 'Andrew Jezard',
 'Nagaraja Uppunda',
 'Raphael Sbarge',
 'Amoghavarsha',
 'Kalyan Varma',
 'Dante Tanikie-Montagnani',
 'Javi Larrauri',
 'Stacey K. Black',
 'Pablo Arévalo',
 'Bonnie Hawthorne',
 'Todd Howe',
 'Agustín Kazah',
 'Maria Bagnat',
 'David Sipos',
 'Will Watson',
 'J.M. Berrios',
 'Steve Wystrach',
 'Bill Suchy',
 'Thomas Veit',
 'Arsel Arumugam',
 'Tobias Frindt',
 'Joe Heslinga',
 'Tyler Chandler',
 'Pavlina Ivanova',
 'Vyacheslav Bihun',
 'Paul Michael Bloodgood',
 'John L Voth',
 'Bart Hölscher',
 'Joe York',
 'Karzan Kardozi',
 'Keli Price',
 'Colonelu Morteni',
 'André Chandelle',
 'Adam Rabinowitz',
 'Julie Simone',
 'Kimberlee Bassford',
 'Abhinav Thakur',
 'Taylor Morden',
 'Roi Maoz',
 'Sudheer Shanbhogue',
 'Jean Griesser',
 'Charles Mattocks',
 'Chi-Yung Chang',
 'Ramez Silyan',
 'D. 

In [ ]:
# --- 4. Calculate average rating per director ---
director_avg_rating = directors_with_ratings.groupby('director_name')['averagerating'].mean()

# --- 5. Define "top directors" (avg rating >= 7.0) ---
top_directors = director_avg_rating[director_avg_rating >= 7.0].index.tolist()

# --- 6. Mark in merged_df whether movie is by a top director ---
merged_df['is_top_director'] = merged_df['director'].isin(top_directors)

In [68]:
director_avg_rating 

director_name
Emre Oran            10.0
Chad Carpenter       10.0
Loreto Di Cesare     10.0
Michiel Brongers     10.0
Masahiro Hayakawa    10.0
                     ... 
Takefumi Tsutsui      1.0
Takeo Urakami         1.0
Kenji Tani            1.0
Shûko Nemoto          1.0
Simon Pennekamp       1.0
Name: averagerating, Length: 56742, dtype: float64

In [55]:
table_names = tables['name'].tolist()

# Step 2: Loop through tables
for table in table_names:
    print(f"\n=== Table: {table} ===\n")
    
    # Show columns
    columns = pd.read_sql_query(f"PRAGMA table_info({table});", conn)
    print("Columns:")
    print(columns[['name', 'type']])  # Just show column name and type for simplicity
    
    # Show first 5 rows
    sample_data = pd.read_sql_query(f"SELECT * FROM {table} LIMIT 5;", conn)
    print("\nSample rows:")
    print(sample_data)
    print("-" * 60)


=== Table: movie_basics ===

Columns:
              name     type
0         movie_id     TEXT
1    primary_title     TEXT
2   original_title     TEXT
3       start_year  INTEGER
4  runtime_minutes     REAL
5           genres     TEXT

Sample rows:
    movie_id                    primary_title              original_title  \
0  tt0063540                        Sunghursh                   Sunghursh   
1  tt0066787  One Day Before the Rainy Season             Ashad Ka Ek Din   
2  tt0069049       The Other Side of the Wind  The Other Side of the Wind   
3  tt0069204                  Sabse Bada Sukh             Sabse Bada Sukh   
4  tt0100275         The Wandering Soap Opera       La Telenovela Errante   

   start_year  runtime_minutes                genres  
0        2013            175.0    Action,Crime,Drama  
1        2019            114.0       Biography,Drama  
2        2018            122.0                 Drama  
3        2018              NaN          Comedy,Drama  
4        2017

In [56]:
rt_reviews

,id,review,rating,fresh,critic,top_critic,publisher,date
0,3,A distinctly gallows take on contemporary fina...,3/5,fresh,PJ Nabarro,0,Patrick Nabarro,"November 10, 2018"
1,3,It's an allegory in search of a meaning that n...,NaN,rotten,Annalee Newitz,0,io9.com,"May 23, 2018"
2,3,... life lived in a bubble in financial dealin...,NaN,fresh,Sean Axmaker,0,Stream on Demand,"January 4, 2018"
3,3,Continuing along a line introduced in last yea...,NaN,fresh,Daniel Kasman,0,MUBI,"November 16, 2017"
4,3,... a perverse twist on neorealism...,NaN,fresh,NaN,0,Cinema Scope,"October 12, 2017"
...,...,...,...,...,...,...,...,...
54427,2000,The real charm of this trifle is the deadpan c...,NaN,fresh,Laura Sinagra,1,Village Voice,"September 24, 2002"
54428,2000,NaN,1/5,rotten,Michael Szymanski,0,Zap2it.com,"September 21, 2005"
54429,2000,NaN,2/5,rotten,Emanuel Levy,0,EmanuelLevy.Com,"July 17, 2005"
54430,2000,NaN,2.5/5,rotten,Christopher Null,0,Filmcritic.com,"September 7, 2003"
